# Homework 8: Deep Learning




In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('torch:', torch.__version__, '| cuda disponible:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch: 2.8.0+cpu | cuda disponible: False


## Modelo



In [2]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=(3, 3), padding=0, stride=1)
        self.pool = nn.MaxPool2d((2, 2))
        self.fc1 = nn.Linear(32 * 99 * 99, 64)
        self.out = nn.Linear(64, 1)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.out(x)
        return x

model = CNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params}')

Total parameters: 20073473


### Q1



In [3]:
import torchsummary
torchsummary.summary(model, input_size=(3, 200, 200))

criterion = nn.BCEWithLogitsLoss()

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
              ReLU-2         [-1, 32, 198, 198]               0
         MaxPool2d-3           [-1, 32, 99, 99]               0
            Linear-4                   [-1, 64]      20,072,512
              ReLU-5                   [-1, 64]               0
            Linear-6                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 21.54
Params size (MB): 76.57
Estimated Total Size (MB): 98.57
----------------------------------------------------------------


## Carga de datos



In [4]:
base_transform = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder('data/data/train', transform=base_transform)
val_dataset = datasets.ImageFolder('data/data/test', transform=base_transform)
print('train:', len(train_dataset), '| val:', len(val_dataset))
print('clases:', train_dataset.classes)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=20, shuffle=False)

train: 800 | val: 201
clases: ['curly', 'straight']


## Entrenamiento (10 épocas, sin aumentación)

In [5]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(val_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f'Epoch {epoch+1}/{num_epochs}, '
          f'Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, '
          f'Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}')

Epoch 1/10, Loss: 0.6665, Acc: 0.6112, Val Loss: 0.6511, Val Acc: 0.6617


Epoch 2/10, Loss: 0.5702, Acc: 0.6787, Val Loss: 0.6332, Val Acc: 0.6318


Epoch 3/10, Loss: 0.5207, Acc: 0.7350, Val Loss: 0.6143, Val Acc: 0.6766


Epoch 4/10, Loss: 0.4773, Acc: 0.7600, Val Loss: 0.6049, Val Acc: 0.6617


Epoch 5/10, Loss: 0.4606, Acc: 0.7550, Val Loss: 0.7307, Val Acc: 0.5672


Epoch 6/10, Loss: 0.3954, Acc: 0.8275, Val Loss: 0.6412, Val Acc: 0.6866


Epoch 7/10, Loss: 0.2844, Acc: 0.8838, Val Loss: 0.8307, Val Acc: 0.6816


Epoch 8/10, Loss: 0.2885, Acc: 0.8788, Val Loss: 0.7052, Val Acc: 0.7114


Epoch 9/10, Loss: 0.1882, Acc: 0.9313, Val Loss: 0.9275, Val Acc: 0.6866


Epoch 10/10, Loss: 0.2585, Acc: 0.8912, Val Loss: 0.8158, Val Acc: 0.6915


### Q3 y Q4



In [6]:
import numpy as np
print('Q3 mediana de training acc:', round(float(np.median(history['acc'])), 3))
print('Q4 std de training loss :', round(float(np.std(history['loss'])), 4))

Q3 mediana de training acc: 0.794
Q4 std de training loss : 0.1462


## Data Augmentation



In [7]:
augmented_transform = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset_aug = datasets.ImageFolder('data/data/train', transform=augmented_transform)
train_loader_aug = DataLoader(train_dataset_aug, batch_size=20, shuffle=True)

history_aug = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset_aug)
    epoch_acc = correct_train / total_train
    history_aug['loss'].append(epoch_loss)
    history_aug['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(val_dataset)
    val_epoch_acc = correct_val / total_val
    history_aug['val_loss'].append(val_epoch_loss)
    history_aug['val_acc'].append(val_epoch_acc)

    print(f'Epoch {epoch+1}/{num_epochs}, '
          f'Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, '
          f'Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}')

Epoch 1/10, Loss: 0.6307, Acc: 0.6600, Val Loss: 0.5963, Val Acc: 0.7114


Epoch 2/10, Loss: 0.5801, Acc: 0.6750, Val Loss: 0.5968, Val Acc: 0.7214


Epoch 3/10, Loss: 0.5374, Acc: 0.7175, Val Loss: 0.5870, Val Acc: 0.7015


Epoch 4/10, Loss: 0.5502, Acc: 0.7125, Val Loss: 0.5696, Val Acc: 0.7264


Epoch 5/10, Loss: 0.5249, Acc: 0.7412, Val Loss: 0.6706, Val Acc: 0.6766


Epoch 6/10, Loss: 0.5192, Acc: 0.7488, Val Loss: 0.5651, Val Acc: 0.7512


Epoch 7/10, Loss: 0.5179, Acc: 0.7350, Val Loss: 0.5753, Val Acc: 0.6866


Epoch 8/10, Loss: 0.4892, Acc: 0.7612, Val Loss: 0.5788, Val Acc: 0.7164


Epoch 9/10, Loss: 0.4964, Acc: 0.7488, Val Loss: 0.5606, Val Acc: 0.6965


Epoch 10/10, Loss: 0.4733, Acc: 0.7688, Val Loss: 0.5474, Val Acc: 0.7264


### Q5 y Q6



In [8]:
print('Q5 media de val loss (10 épocas con aumentación):', round(float(np.mean(history_aug['val_loss'])), 3))
print('Q6 promedio de val acc (épocas 6-10):', round(float(np.mean(history_aug['val_acc'][5:])), 3))

Q5 media de val loss (10 épocas con aumentación): 0.585
Q6 promedio de val acc (épocas 6-10): 0.715
